# Merging the Energy and Temperature Datasets
This notebook runs top to bottom. It reads `data/energy_cleaned.csv` (from `ENERGY.ipynb`) and `cleaned_temperature.csv` (from `TEMPERATURE.ipynb`) and writes `energy_breakdown_final.csv` and `merged_final.csv`. It does not overwrite its own inputs.

In [1]:
# Import necessary libraries
import pandas as pd

In [2]:
# Read the cleaned datasets and standardise column names
# (strip spaces, replace spaces with underscores, lowercase)
energy = pd.read_csv("data/energy_cleaned.csv")
energy.columns = energy.columns.str.strip().str.replace(" ", "_").str.lower()

temperature = pd.read_csv("data/cleaned_temperature.csv")
temperature.columns = temperature.columns.str.strip().str.replace(" ", "_").str.lower()

print("Energy columns:", energy.columns.tolist())
print("Temperature columns:", temperature.columns.tolist())

Energy columns: ['country', 'energy_type', 'year', 'energy_consumption', 'energy_production', 'gdp', 'population', 'energy_intensity_per_capita', 'energy_intensity_by_gdp', 'co2_emission']
Temperature columns: ['country', 'year', 'annual_mean', 'five_year_smooth', 'code']


## Standardise country names

In [3]:
# Replace country names in the temperature dataset to match those in the energy dataset,
# and to use names Tableau recognises.
temperature["country"] = temperature["country"].replace({
    "Russian Federation": "Russia",
    "Korea, Rep.": "South Korea",
    "Korea-": "North Korea",
    "Iran, Islamic Rep.": "Iran",
    "Congo, Dem. Rep.": "Congo-Kinshasa",
    "Gambia,": "Gambia, The",
    "Guinea Bissau": "Guinea-Bissau",
    "Kyrgyz Republic": "Kyrgyzstan",
    "Myanmar": "Burma",
    "Slovak Republic": "Slovakia",
    "Syrian Arab Republic": "Syria",
    "Timor Leste": "Timor-Leste",
    "Trinidad And Tobago": "Trinidad and Tobago",
    "Venezuela, Rb": "Venezuela",
    "Yemen, Rep.": "Yemen",
    "Bahamas,": "The Bahamas",
    "Cote d'Ivoire": "Côte d’Ivoire",
})

# The source file lists Gambia twice ("Gambia" and "Gambia,"). Keep one copy only,
# otherwise the merge would duplicate Gambia's rows.
temperature = temperature[temperature["country"] != "Gambia"]
assert not temperature.duplicated(["country", "year"]).any()

In [4]:
# Remove historical / dissolved countries (Former U.S.S.R., Former Yugoslavia, ...).
# They don't exist in modern geographic datasets, so they can't be merged with the
# temperature data or mapped in Tableau.
historical = [
    "Former U.S.S.R.",
    "Former Yugoslavia",
    "Former Czechoslovakia",
    "Former Serbia and Montenegro",
    "Germany, East",
    "Germany, West",
    "Hawaiian Trade Zone",
    "Netherlands Antilles",
]
# Remove non-country entities that Tableau doesn't recognise.
non_countries = [
    "U.S. Pacific Islands",
    "U.S. Territories",
    "Wake Island",
]
energy = energy[~energy["country"].isin(historical + non_countries)]

# Use Tableau's spelling for Saint Vincent and the Grenadines
energy["country"] = energy["country"].replace({
    "Saint Vincent/Grenadines": "Saint Vincent and the Grenadines"
})
print(f"Countries remaining: {energy['country'].nunique()}")

Countries remaining: 202


## Merge

In [5]:
# Left join: keep every energy record, even where temperature data is missing
merged = pd.merge(
    energy[energy["energy_type"] == "all_energy_types"],
    temperature,
    on=["country", "year"],
    how="left"
)
# Store year as a plain integer (1980 instead of 1980-01-01) for Tableau
merged["year"] = pd.to_datetime(merged["year"]).dt.year

print(f"Merged rows: {len(merged)}")
print(f"Countries: {merged['country'].nunique()}")
print(f"Year range: {merged['year'].min()} - {merged['year'].max()}")
print(f"CO2 nulls: {merged['co2_emission'].isna().sum()}")
print(f"Rows without temperature: {merged['annual_mean'].isna().sum()}")

Merged rows: 8080
Countries: 202
Year range: 1980 - 2019
CO2 nulls: 648
Rows without temperature: 1560


In [6]:
# Countries with energy data but no temperature data.
# Note: China and India are in this list. They are not in the temperature source
# file at all, so no renaming can fix this; they have no temperature values.
no_temp = sorted(merged.loc[merged["annual_mean"].isna(), "country"].unique())
print(len(no_temp), "countries without temperature data:")
print(no_temp)

39 countries without temperature data:
['American Samoa', 'Antigua and Barbuda', 'Aruba', 'Bermuda', 'Bosnia and Herzegovina', 'British Virgin Islands', 'Brunei', 'Burkina Faso', 'Cabo Verde', 'Cayman Islands', 'China', 'Congo-Brazzaville', 'Czech Republic', 'El Salvador', 'Equatorial Guinea', 'Greenland', 'Guadeloupe', 'Guam', 'Hong Kong', 'India', 'Ireland', 'Kosovo', 'Laos', 'Lithuania', 'Macau', 'Martinique', 'Morocco', 'Namibia', 'North Macedonia', 'Northern Mariana Islands', 'Palestinian Territories', 'Puerto Rico', 'Reunion', 'Saint Kitts and Nevis', 'Saint Lucia', 'Saint Vincent and the Grenadines', 'Sao Tome and Principe', 'Taiwan', 'U.S. Virgin Islands']


## Quick checks
The CO2 trend is only meaningful in years where most countries report CO2. In this dataset only the United States has a CO2 value for 1980, so a raw world total for the early 1980s is really just the US.

In [7]:
# How many countries report CO2 each year?
all_types = merged.copy()
coverage = all_types.groupby("year")["co2_emission"].apply(lambda s: (s > 0).sum())
coverage.head(12)

year
1980      1
1981    169
1982    169
1983    169
1984    169
1985    169
1986    169
1987    170
1988    170
1989    170
1990    170
1991    171
Name: co2_emission, dtype: int64

In [8]:
# Global CO2 total, only for years where at least 90% of the best-covered year's
# countries report CO2
complete_years = coverage[coverage >= 0.9 * coverage.max()].index
co2_by_year = all_types.groupby("year")["co2_emission"].sum()
print(f"Years with near-complete CO2 coverage: {complete_years.min()} - {complete_years.max()}")
co2_by_year.loc[complete_years]

Years with near-complete CO2 coverage: 1993 - 2019


year
1993    21496.757901
1994    21817.259660
1995    21992.542056
1996    22470.558160
1997    22953.741425
1998    23017.828766
1999    23055.224836
2000    23415.496216
2001    24171.160618
2002    24378.559900
2003    24926.391920
2004    26075.619573
2005    27505.346317
2006    28598.212428
2007    29607.456155
2008    30061.492882
2009    30771.351511
2010    30596.497098
2011    32484.738398
2012    33596.769409
2013    34387.882664
2014    34805.003484
2015    34859.302551
2016    34714.645861
2017    34534.264836
2018    34963.579486
2019    35545.134531
Name: co2_emission, dtype: float64

In [9]:
# Which countries emitted the most CO2 in total (1980-2019)?
merged.groupby("country")["co2_emission"].sum().sort_values(ascending=False).head(10)

country
United States     211603.865978
China             197778.559462
Russia             45265.267036
Japan              44240.174272
India              39905.319466
Germany            24045.492716
United Kingdom     21626.383620
Canada             19625.257692
South Korea        16441.794472
Italy              15847.811774
Name: co2_emission, dtype: float64

In [10]:
# Largest emitter in each year: when did China pass the United States?
leader = merged.loc[merged.groupby("year")["co2_emission"].idxmax().dropna(), ["year", "country"]]
leader[leader["country"] != leader["country"].shift()]

,year,country
193,1980,United States
5291,2006,China


## Save

In [11]:
# Save the final datasets. energy_cleaned.csv is NOT overwritten, so the
# notebooks give the same result however many times they are run.
energy.to_csv("data/energy_breakdown_final.csv", index=False)
merged.to_csv("data/merged_final.csv", index=False)
print("Saved successfully")

Saved successfully
